# 08. Multi-Task Models

## Purpose
Notebooks 06 and 07 each predict a *single* outcome in isolation. But injury
risk is really a bundle of related questions:

- **Will** this pitcher be injured (30 / 60 / 90 days)?
- **When** will it happen (`days_to_next_injury`)?
- **How severe** will it be (`next_injury_days_lost`)?
- **What kind** of injury (`next_injury_type`)?

These tasks share an underlying cause, accumulated workload and mechanical
strain, so a model that learns them jointly can borrow statistical strength
from the more common tasks (binary injury within 30d) to improve the rarer,
harder ones (severity, type). This notebook compares two multi-task
architectures:

1. **Chained model**: predicts injury probability first, then feeds that
   prediction as an input feature to the downstream severity/type/timing heads
   (reflects the causal ordering: *whether* precedes *how bad*)
2. **Shared-representation model**: a single fitted preprocessing trunk
   feeds independent per-task heads (the practical analogue of hard parameter
   sharing for heterogeneous task types with tree ensembles)

## Why this matters for Injury Risk+
The composite score (notebook 09) needs `injury_prob_30d`,
`expected_days_lost`, and a hazard signal *simultaneously, for the same
pitcher*. Multi-task models are a natural way to produce a coherent bundle
of these predictions from one fitting pass.

In [ ]:
import sys, json, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

TEST_MODE = False  # True = 5 000-row sample for fast iteration; False = full production run

PROJECT_ROOT = str(Path('.').resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.baseline_models import _infer_feature_cols, save_model
from src.models.multitask_models import (
    prepare_multitask_dataset,
    train_chained_multitask_model,
    train_shared_representation_model,
    predict_all_tasks,
    compute_multitask_metrics,
    ALL_TASKS,
    CLASSIFICATION_TASKS,
    REGRESSION_TASKS,
)

MODELS_DIR  = Path('models')
TABLES_DIR  = Path('reports/tables')
FIGURES_DIR = Path('reports/figures')
for d in (MODELS_DIR, TABLES_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'TEST_MODE={TEST_MODE}  Modules loaded. Tasks: {ALL_TASKS}')

## 1. Load Feature Matrix and Build the Multi-Task Dataset

`prepare_multitask_dataset` builds a single shared temporal `(X_train, X_test)`
split, holding out the most recent seasons, plus a dictionary of target arrays
per task. The regression and multiclass targets `days_to_next_injury`,
`next_injury_days_lost`, and `next_injury_type` are only observed for pitchers
who were actually later injured, everyone else is `NaN`, and the training
routines filter to the observed subset internally via `_regression_subset`.

In [ ]:
fm = pd.read_parquet('data/processed/feature_matrix.parquet')
fm['game_date'] = pd.to_datetime(fm['game_date'])
seasons = sorted(fm['season'].unique().tolist())

if TEST_MODE:
    fm = fm.sample(n=min(5_000, len(fm)), random_state=42).copy()
    print(f'TEST_MODE: subsampled to {len(fm):,} rows')

feature_cols = _infer_feature_cols(fm)
X_train, X_test, y_train_dict, y_test_dict = prepare_multitask_dataset(fm, feature_cols=feature_cols)

print(f'Seasons: {seasons}')
print(f'Train: {len(X_train):,} rows | Test: {len(X_test):,} rows | Features: {len(feature_cols)}')
print()
print('Task observation rates (train):')
for task in ALL_TASKS:
    n_obs = int(y_train_dict[task].notna().sum())
    print(f'  {task:24s}  observed = {n_obs:5,d} / {len(y_train_dict[task]):,}  '
          f'({n_obs / len(y_train_dict[task]):.1%})')

## 2. Train Both Architectures

In [ ]:
%%time
print('Training chained multi-task model...')
chained_model = train_chained_multitask_model(X_train, y_train_dict)

print('Training shared-representation model...')
shared_model = train_shared_representation_model(X_train, y_train_dict, shared_model_type='gradient_boosting')

print('\nBoth architectures trained.')

## 3. Evaluate Both Architectures on the Held-Out Test Set

For each architecture we run inference for every task in one pass with
`predict_all_tasks` and score with `compute_multitask_metrics`:
classification tasks get AUC-ROC / PR-AUC / Brier, regression tasks get
MAE / RMSE / R², and the multiclass task gets accuracy / macro-F1.

In [ ]:
chained_preds = predict_all_tasks(chained_model, X_test)
shared_preds  = predict_all_tasks(shared_model, X_test)

chained_metrics = compute_multitask_metrics(y_test_dict, chained_preds)
shared_metrics  = compute_multitask_metrics(y_test_dict, shared_preds)

print('Chained multi-task model — test metrics:')
display(chained_metrics)
print()
print('Shared-representation model — test metrics:')
display(shared_metrics)

## 3b. Multi-Task Model Hyperparameter Tuning

We tune the **primary classification head** for 30-day binary injury and the
**days-lost regression head** independently, since they have the most impact
on the final Injury Risk+ score. Severity/type heads are tuned when
observation counts allow.

`FAST_TUNING = True` keeps search spaces laptop-safe.

In [ ]:
RUN_TUNING    = True
FAST_TUNING   = True
N_ITER_TUNING = 5 if TEST_MODE else 20

MT_TUNING_CHECKPOINT = TABLES_DIR / 'multitask_hyperparameter_tuning_results.csv'
print(f'RUN_TUNING={RUN_TUNING}  FAST_TUNING={FAST_TUNING}  N_ITER={N_ITER_TUNING}  TEST_MODE={TEST_MODE}')

In [ ]:
%%time
import joblib
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, mean_absolute_error, f1_score
from src.models.multitask_models import _regression_subset

mt_tuning_records = []
chained_tuned = None

if RUN_TUNING:
    _n_est = [100, 200] if FAST_TUNING else [200, 400, 600]

# Tune classification head (injured_next_30d)
    print('Tuning classification head (injured_next_30d)...')
    _clf_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(random_state=42, n_jobs=-1)),
    ])
    _clf_space = {
        'clf__n_estimators': _n_est,
        'clf__max_depth': [4, 6, 8, None],
        'clf__min_samples_split': [2, 5, 10],
        'clf__min_samples_leaf': [1, 3, 5],
        'clf__max_features': ['sqrt', 'log2'],
        'clf__class_weight': ['balanced', 'balanced_subsample'],
    }
    _clf_cv = RandomizedSearchCV(
        _clf_pipe, _clf_space, n_iter=N_ITER_TUNING,
        scoring='average_precision', cv=3, random_state=42, n_jobs=-1, refit=True,
    )
    _clf_cv.fit(X_train, y_train_dict['injured_next_30d'])
    _best_clf = _clf_cv.best_estimator_
    _clf_pr_auc = average_precision_score(
        y_test_dict['injured_next_30d'],
        _best_clf.predict_proba(X_test)[:, 1],
    )
    mt_tuning_records.append({
        'task': 'injured_next_30d', 'type': 'classification', 'metric': 'pr_auc',
        'cv_score': _clf_cv.best_score_, 'test_score': _clf_pr_auc,
        'best_params': str(_clf_cv.best_params_),
    })
    print(f'  CV PR AUC = {_clf_cv.best_score_:.4f}  | Test PR AUC = {_clf_pr_auc:.4f}')

# Tune regression head (next_injury_days_lost)
    # Must include injury_prob_30d so the tuned head matches what predict_all_tasks
    # will pass at inference time (the chained feature is added before regression).
    print('Tuning regression head (next_injury_days_lost)...')
    _X_train_chained = X_train.copy()
    _X_train_chained['injury_prob_30d'] = _best_clf.predict_proba(X_train)[:, 1]
    _X_test_chained = X_test.copy()
    _X_test_chained['injury_prob_30d'] = _best_clf.predict_proba(X_test)[:, 1]

    X_sub_reg, y_sub_reg = _regression_subset(_X_train_chained, y_train_dict['next_injury_days_lost'])
    X_sub_test, y_sub_test = _regression_subset(_X_test_chained, y_test_dict['next_injury_days_lost'])

    if len(y_sub_reg) >= 20:
        _reg_pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('reg', RandomForestRegressor(random_state=42, n_jobs=-1)),
        ])
        _reg_space = {
            'reg__n_estimators': _n_est,
            'reg__max_depth': [4, 6, 8, None],
            'reg__min_samples_split': [2, 5, 10],
            'reg__min_samples_leaf': [3, 5, 10],
            'reg__max_features': ['sqrt', 'log2'],
        }
        _reg_cv = RandomizedSearchCV(
            _reg_pipe, _reg_space, n_iter=N_ITER_TUNING,
            scoring='neg_mean_absolute_error', cv=3, random_state=42, n_jobs=-1, refit=True,
        )
        _reg_cv.fit(X_sub_reg, y_sub_reg)
        _best_reg = _reg_cv.best_estimator_
        _reg_mae = mean_absolute_error(y_sub_test, _best_reg.predict(X_sub_test)) if len(y_sub_test) > 0 else float('nan')
        mt_tuning_records.append({
            'task': 'next_injury_days_lost', 'type': 'regression', 'metric': 'mae',
            'cv_score': -_reg_cv.best_score_, 'test_score': _reg_mae,
            'best_params': str(_reg_cv.best_params_),
        })
        print(f'  CV MAE = {-_reg_cv.best_score_:.1f}  | Test MAE = {_reg_mae:.1f}')
    else:
        _best_reg = None
        print(f'  Skipped (only {len(y_sub_reg)} observed rows)')

# Build tuned chained model
    print('Building tuned chained multitask model...')
    chained_tuned = dict(chained_model)  # copy existing heads
    chained_tuned['injured_next_30d'] = _best_clf
    if _best_reg is not None:
        chained_tuned['next_injury_days_lost'] = _best_reg
    # Rebuild 60d, 90d heads with same best params
    for t in ['injured_next_60d', 'injured_next_90d']:
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('clf', RandomForestClassifier(random_state=42, n_jobs=-1,
                                           **{k.replace('clf__', ''): v
                                              for k, v in _clf_cv.best_params_.items()})),
        ])
        pipe.fit(X_train, y_train_dict[t])
        chained_tuned[t] = pipe
        test_pr = average_precision_score(
            y_test_dict[t], pipe.predict_proba(X_test)[:, 1])
        mt_tuning_records.append({
            'task': t, 'type': 'classification', 'metric': 'pr_auc',
            'cv_score': None, 'test_score': test_pr,
            'best_params': str(_clf_cv.best_params_),
        })
        print(f'  {t}: Test PR AUC = {test_pr:.4f}')

    mt_tuning_df = pd.DataFrame(mt_tuning_records)
    display(mt_tuning_df[['task', 'type', 'metric', 'cv_score', 'test_score']])

else:
    mt_tuning_df = pd.read_csv(MT_TUNING_CHECKPOINT) if MT_TUNING_CHECKPOINT.exists() else pd.DataFrame()
    print('Skipped tuning (RUN_TUNING=False)')

print(f'\nTuned chained model ready: {chained_tuned is not None}')

## 4. Head-to-Head Comparison

Side-by-side AUC-ROC for classification tasks and MAE for regression tasks
across both architectures, showing which one wins and on which kinds of tasks.

In [ ]:
compare_rows = []
for task in CLASSIFICATION_TASKS:
    if task in chained_metrics.index and task in shared_metrics.index:
        compare_rows.append({
            'task': task, 'metric': 'auc_roc',
            'chained': chained_metrics.loc[task, 'auc_roc'],
            'shared': shared_metrics.loc[task, 'auc_roc'],
        })
for task in REGRESSION_TASKS:
    if task in chained_metrics.index and task in shared_metrics.index:
        compare_rows.append({
            'task': task, 'metric': 'mae',
            'chained': chained_metrics.loc[task, 'mae'],
            'shared': shared_metrics.loc[task, 'mae'],
        })

compare_df = pd.DataFrame(compare_rows).set_index(['task', 'metric'])
compare_df['winner'] = np.where(
    compare_df.index.get_level_values('metric') == 'mae',
    np.where(compare_df['chained'] < compare_df['shared'], 'chained', 'shared'),
    np.where(compare_df['chained'] > compare_df['shared'], 'chained', 'shared'),
)
display(compare_df.style.format({'chained': '{:.3f}', 'shared': '{:.3f}'}))

n_chained_wins = int((compare_df['winner'] == 'chained').sum())
n_shared_wins = int((compare_df['winner'] == 'shared').sum())
best_architecture = 'chained' if n_chained_wins >= n_shared_wins else 'shared'
print(f'\nChained wins {n_chained_wins}/{len(compare_df)} comparisons; '
      f'Shared wins {n_shared_wins}/{len(compare_df)}.')
print(f'Selected architecture for downstream use: {best_architecture}')

## 5. Visualize Task Performance

A compact view of how well each architecture handles each task family:
classification measured by AUC-ROC where higher is better, versus regression
measured by MAE where lower is better and shown as inverted bars for visual
consistency.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
_metric_levels = compare_df.index.get_level_values('metric').unique()

clf_compare = compare_df.xs('auc_roc', level='metric') if 'auc_roc' in _metric_levels else pd.DataFrame(columns=['chained', 'shared'])
ax = axes[0]
x = np.arange(len(clf_compare))
width = 0.35
ax.bar(x - width / 2, clf_compare['chained'], width, label='chained', color='#4C72B0')
ax.bar(x + width / 2, clf_compare['shared'], width, label='shared', color='#C44E52')
ax.set_xticks(x)
ax.set_xticklabels(clf_compare.index, rotation=20, ha='right')
ax.set_ylabel('AUC-ROC')
ax.set_title('Classification tasks (higher is better)')
ax.legend()

reg_compare = compare_df.xs('mae', level='metric') if 'mae' in _metric_levels else pd.DataFrame(columns=['chained', 'shared'])
ax = axes[1]
if len(reg_compare) > 0:
    x = np.arange(len(reg_compare))
    ax.bar(x - width / 2, reg_compare['chained'], width, label='chained', color='#4C72B0')
    ax.bar(x + width / 2, reg_compare['shared'], width, label='shared', color='#C44E52')
    ax.set_xticks(x)
    ax.set_xticklabels(reg_compare.index, rotation=20, ha='right')
    ax.legend()
else:
    ax.text(0.5, 0.5, 'No regression tasks with\nsufficient observations',
            ha='center', va='center', transform=ax.transAxes, fontsize=11, color='gray')
ax.set_ylabel('MAE (days)')
ax.set_title('Regression tasks (lower is better)')

fig.tight_layout()
fig_path = FIGURES_DIR / 'fig_23_multitask_comparison.png'
fig.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')

## 6. Save Models and Results

We persist both architectures so notebook 09 can choose either, but flag the
winner from the head-to-head comparison as the default for Injury Risk+
inference.

In [ ]:
import joblib

chained_path = MODELS_DIR / 'multitask_chained.joblib'
shared_path  = MODELS_DIR / 'multitask_shared.pkl'
joblib.dump(chained_model, chained_path)
joblib.dump(shared_model, shared_path)
print(f'Saved {chained_path}')
print(f'Saved {shared_path}')

if chained_tuned is not None:
    tuned_path = MODELS_DIR / 'multitask_chained_tuned.joblib'
    joblib.dump(chained_tuned, tuned_path)
    print(f'Saved {tuned_path}')

chained_metrics.assign(architecture='chained').reset_index().to_csv(
    TABLES_DIR / 'multitask_chained_metrics.csv', index=False)
shared_metrics.assign(architecture='shared').reset_index().to_csv(
    TABLES_DIR / 'multitask_shared_metrics.csv', index=False)
compare_df.reset_index().to_csv(TABLES_DIR / 'multitask_comparison.csv', index=False)

# Combined metrics table
pd.concat([
    chained_metrics.assign(architecture='chained'),
    shared_metrics.assign(architecture='shared'),
]).reset_index().to_csv(TABLES_DIR / 'multitask_model_metrics.csv', index=False)
print(f'Saved {TABLES_DIR / "multitask_model_metrics.csv"}')

if mt_tuning_records:
    pd.DataFrame(mt_tuning_records).to_csv(MT_TUNING_CHECKPOINT, index=False)
    print(f'Saved {MT_TUNING_CHECKPOINT}')

if chained_tuned is not None:
    tuned_preds = predict_all_tasks(chained_tuned, X_test)
    tuned_metrics = compute_multitask_metrics(y_test_dict, tuned_preds)
    tuned_metrics.assign(architecture='chained_tuned').reset_index().to_csv(
        TABLES_DIR / 'tuned_multitask_model_metrics.csv', index=False)
    print(f'Saved {TABLES_DIR / "tuned_multitask_model_metrics.csv"}')

## 7. Summary

* **Architecture comparison:** see section 4. Whichever model wins more
  task-level comparisons becomes the default multi-task model fed into
  Injury Risk+ (notebook 09), though both are saved to `models/` for
  flexibility.
* **Task difficulty gradient:** classification tasks (binary injury within N
  days) are the best-observed and easiest; regression/multiclass tasks
  (severity, timing, type) are trained on a much smaller "injured-only"
  subset and are correspondingly noisier, exactly the rare-outcome problem
  multi-task learning is meant to help with.
* **Next step:** notebook 09 combines `injury_prob_30d`,
  `expected_days_lost`, and a survival-derived `hazard_rate` into the
  composite Injury Risk+ score, calibrates it, and produces the final
  pitcher rankings.

In [ ]:
provenance = {
    'notebook': '08_multitask_models',
    'run_at': datetime.now(timezone.utc).isoformat(),
    'seasons_used': seasons,
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'n_features': len(feature_cols),
    'best_architecture': best_architecture,
    'n_chained_wins': n_chained_wins,
    'n_shared_wins': n_shared_wins,
    'chained_metrics': chained_metrics.to_dict(orient='index'),
    'shared_metrics': shared_metrics.to_dict(orient='index'),
}
print(json.dumps(provenance, indent=2, default=str))

prov_path = TABLES_DIR / 'multitask_model_provenance.json'
prov_path.write_text(json.dumps(provenance, indent=2, default=str))
print(f'\nSaved {prov_path}')